# SN PB-002 
## Adding annotation

kernel: scvi-r  docker kernel is used `docker pull alexanrna/scanpy-r:1.10-v6`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
import anndata
from scipy import stats
import glob 
import os
from matplotlib import pyplot as plt
import datetime
from scipy.sparse import csr_matrix
import warnings
import warnings
import anndata as ad
import re
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


# Load data

In [ ]:
adata =  sc.read_h5ad('./matrices/20250209_all_PB-002_sn.h5ad')

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
np.unique(adata.X.data)

# Recluster data

In [ ]:
#adata.X = adata.layers["counts"].copy()
print(np.unique(adata.X.data))
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
sc.pl.pca_scatter(adata, color="total_counts")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

## Load annotations from scANVI

In [ ]:
# Load the CSV file into a dictionary
df = pd.read_csv('./scANVI/20250210_PB-002_celltype_from_scANVI_sn.csv')
data_dict = df.set_index('index')['column_name'].to_dict()
adata.obs['cell_type_scANVI'] = adata.obs.index.map(data_dict)
sc.pl.umap(
    adata,
    color="cell_type_scANVI", # Add any other columns here
    frameon=False,
)

# Remove doublets

In [ ]:
adata.obs.drop(columns=['leiden_1.2','leiden_1.5'])

In [ ]:
res = 4.5
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(adata, color="leiden_4.5",)

In [ ]:
sc.pl.umap(adata, color="vireo_assignment", groups = 'doublet')
sc.pl.umap(adata, color="vireo_assignment", groups = 'unassigned')

In [ ]:
# what is fraction of doublets per cluster

# init lists
list_clusters = np.unique(adata.obs['leiden_4.5'])
list_size = []
list_frac_doublets = []

# loop over clusters
for clus in list_clusters:
    clus_adata = adata[adata.obs['leiden_4.5'] == clus]
    
    size = clus_adata.shape[0]
    n_doublets = sum( clus_adata.obs['vireo_assignment'] == 'doublet')
    
    list_frac_doublets.append( n_doublets / size )
    list_size.append(size)
    
df_frac_doublets = pd.DataFrame(
{
    'cluster': list_clusters,
    'size': list_size,
    'frac_doublets': list_frac_doublets
}
)

In [ ]:
df_frac_doublets.sort_values('frac_doublets', ascending=False)

In [ ]:
doublet_clusters = [ clus for clus 
                    in df_frac_doublets[ df_frac_doublets['frac_doublets'] >= 0.3 ]['cluster'] ]
doublet_clusters

In [ ]:
# remove doublet clusters, doublets and unassigned
adata = adata[ ~adata.obs['leiden_4.5'].isin(doublet_clusters) ].copy()
adata = adata[ adata.obs['vireo_assignment'] != 'doublet' ].copy()
adata = adata[ adata.obs['vireo_assignment'] != 'unassigned' ].copy()
adata

# Check marker genes

In [ ]:
marker_genes = {
    "neurons": ["SYT1"],
    "OLG": ["MAG", "MOBP"],
    "OPC": [
        "PDGFRA"],
    "Ast":["AQP4", "GFAP"],
    "Micro":["CD74","RUNX1"],
    "Endo":["CLDN5"]}

SN = {
    "neurons",
    "OLG",
    "OPC",
    "Ast",
    "Micro",
    "Endo"}

marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

for ct in SN:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.umap(
        adata,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax="p99",  
        sort_order=False,  
        frameon=False,
        cmap="Reds", 
    )
    print("\n\n\n")  

In [ ]:
adata.obs.cell_type_scANVI.value_counts()

In [ ]:
adata.obs

# recluster the data

In [ ]:
adata.X = adata.layers["counts"].copy()
print(np.unique(adata.X.data))
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
sc.pl.pca_scatter(adata, color="total_counts")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color="cell_type_scANVI", # Add any other columns here
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata,
    color="vireo_assignment",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata,
    color="MALAT1",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "pct_counts_mt"]
)

In [ ]:
adata.obs

## Add metadata

In [ ]:
def remove_trailing_letter(subject_id):
    if subject_id.endswith('A') or subject_id.endswith('B') or subject_id.endswith('C') or subject_id.endswith('D'):
        return subject_id[:-1]
    return subject_id

In [ ]:
# add metadata
metadata_path = '/donor_statistics/crn_metadata/SUBJECT.csv' # change this to the correct path of the metadata file as necessary
metadata = pd.read_csv(metadata_path)

metadata['subject_id'] = metadata['subject_id'].apply(remove_trailing_letter)
metadata_cleaned = metadata.drop_duplicates()

columns_to_map = ['biobank_name','sex','age_at_collection','primary_diagnosis','age_at_diagnosis']
mapping_dicts = {col: metadata_cleaned.set_index('subject_id')[col].to_dict() for col in columns_to_map}


for col, mapping_dict in mapping_dicts.items():
    adata.obs[col] = adata.obs['vireo_assignment'].map(mapping_dict)

adata.obs

In [ ]:
for c in columns_to_map:
    sc.pl.umap(
    adata,
    color=c,
)

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="cell_type_scANVI", method="wilcoxon", key_added="dea_cell_type_scANVI"
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="cell_type_scANVI", standard_scale="var", n_genes=5, key="dea_cell_type_scANVI", dendrogram = True
)

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')
adata.X = adata.layers['counts'].copy()
adata.write(f'./matrices/{date}_PB-002_filtered_umap_tsne_annotated_scANVI_metadata.sn.h5ad')

In [ ]:
import session_info

session_info.show()

In [ ]:
pip list